# File Handling - Paths With `pathlib`

`pathlib` represents file system paths as **objects** instead of plain strings. It works the same on Windows, macOS and Linux.

| Member | Purpose |
|---|---|
| `Path("data/file.txt")` | Create a path |
| `Path.cwd()` / `Path.home()` | Current folder / home folder |
| `p / "sub" / "file.txt"` | Join parts with `/` |
| `p.name` / `p.stem` / `p.suffix` | Full file name / name without the last extension / last extension |
| `p.parent` / `p.parents` | Containing folder(s) |
| `p.with_name()` / `p.with_suffix()` | Same path with a different name or extension |
| `p.exists()` / `p.is_file()` / `p.is_dir()` | Check what exists |
| `p.mkdir(parents=True, exist_ok=True)` | Create a folder |
| `p.iterdir()` | List the items in a folder |
| `p.glob("*.txt")` / `p.rglob("*.txt")` | Pattern search (`rglob` searches subfolders) |
| `p.read_text()` / `p.write_text()` | Read or write a whole text file |
| `p.read_bytes()` / `p.write_bytes()` | Read or write binary data |
| `p.rename()` / `p.unlink()` / `p.rmdir()` | Rename or delete |
| `p.resolve()` | Absolute path |
| `p.relative_to(base)` | Path relative to another path |

---

## Creating and Joining Paths

```python
from pathlib import Path

base = Path("project")
file = base / "data" / "notes.txt"
```

The `/` operator joins parts with the correct separator for the operating system.

---

## Parts of a Path

| Property | For `project/data/report.final.csv` |
|---|---|
| `name` | `report.final.csv` |
| `stem` | `report.final` |
| `suffix` | `.csv` |
| `suffixes` | `['.final', '.csv']` |
| `parent` | `project/data` |
| `parts` | `('project', 'data', 'report.final.csv')` |

---

## Reading and Writing

```python
path.write_text("hello", encoding="utf-8")
path.read_text(encoding="utf-8")
```

For large files or line-by-line work, use `path.open()`, exactly like `open()`.

---

## Folders

```python
folder.mkdir(parents=True, exist_ok=True)
for item in folder.iterdir():
    ...
list(folder.glob("*.txt"))      # in this folder only
list(folder.rglob("*.txt"))     # in all subfolders
```

* `parents=True` creates missing parent folders.
* `exist_ok=True` avoids an error if the folder already exists.
* `iterdir()`, `glob()` and `rglob()` return iterators. The order is not guaranteed, so use `sorted()`.

---

## `pathlib` vs `os.path`

| Task | `os.path` | `pathlib` |
|---|---|---|
| Join | `os.path.join(a, "b")` | `a / "b"` |
| File name | `os.path.basename(p)` | `p.name` |
| Extension | `os.path.splitext(p)[1]` | `p.suffix` |
| Exists | `os.path.exists(p)` | `p.exists()` |
| Read a file | `open(p).read()` | `p.read_text()` |

### Important

* Prefer `pathlib` for new code.
* Python 3.12 adds `Path.walk()` for walking a folder tree.
* Deleting is permanent. `unlink()` and `rmdir()` do not use a recycle bin.

## Source

https://docs.python.org/3/library/pathlib.html

In [ ]:
import tempfile
from pathlib import Path

with tempfile.TemporaryDirectory() as tmp:
    base = Path(tmp) / "project"
    data = base / "data"
    data.mkdir(parents=True, exist_ok=True)               # creates project/ and project/data/
    print(data.is_dir(), data.exists(), (data / "x").exists())

    # Joining and parts of a path
    report = data / "report.final.csv"
    print(report.name, "|", report.stem, "|", report.suffix, "|", report.suffixes)
    print(report.parent.name, report.parts[-3:])
    print(report.with_suffix(".txt").name, report.with_name("other.csv").name)
    print(report.relative_to(base))

    # Writing and reading
    notes = data / "notes.txt"
    notes.write_text("first line\nsecond line\n", encoding="utf-8")
    print(notes.read_text(encoding="utf-8").splitlines())
    print(notes.stat().st_size, notes.is_file())

    # Line by line with open()
    with notes.open(encoding="utf-8") as handle:
        print([line.strip() for line in handle])

    # Create more files, then search
    (data / "a.txt").write_text("a")
    (data / "b.csv").write_text("b")
    sub = data / "archive"
    sub.mkdir()
    (sub / "old.txt").write_text("old")

    print(sorted(p.name for p in data.iterdir()))
    print(sorted(p.name for p in data.glob("*.txt")))       # this folder only
    print(sorted(p.name for p in data.rglob("*.txt")))      # including subfolders

    # Rename and delete
    renamed = (data / "a.txt").rename(data / "renamed.txt")
    print(renamed.name, (data / "a.txt").exists())
    renamed.unlink()
    (data / "missing.txt").unlink(missing_ok=True)          # no error if it does not exist
    print(sorted(p.name for p in data.iterdir()))

# The folder and everything in it are gone
print(Path(tmp).exists())

# Home and current folder exist as paths
print(Path.home().is_absolute(), Path.cwd().is_dir())